# Anotador TDAH · 01/04 · Backend `directo`

El modelo hace un `POST` a `/api/generate` de Ollama con el prompt, y extraer el JSON de la respuesta de texto. Nada obliga al modelo a devolver JSON válido: si devuelve la respuesta con explicaciones o en ` ```json `, `extraer_json` intenta rescatarlo, y si no puede, la anotación cuenta como **fallo de formato**.

Es la **línea base**: cualquier otro backend debería igualar o superar su tasa de formato válido.


## 1 · Parámetros

In [1]:
import datetime as dt
import json
import sqlite3
import time

import pandas as pd

# --- Parámetros del experimento ---
SEMANA       = 1            # semana de seguimiento (el dataset llega a la 24)
PACIENTES    = None         # None = todos los de la semana; o lista: ["P001", "P003"]
REPETICIONES = 3            # veces que se anota cada entrada
TEMPERATURA  = 0.7
MODELO       = "gemma4:e4b" 

# --- Rutas y conexión ---
RUTA_BD     = "datos/anotador.db"
OLLAMA_URL  = "http://127.0.0.1:11002"   
INSTRUMENTO = "instrumentos/brief2.json"

BACKEND     = "directo"
EXPERIMENTO = f"{BACKEND}-s{SEMANA}-t{TEMPERATURA}-{dt.date.today():%Y%m%d}"
print(f"Código de experimento: {EXPERIMENTO}")

Código de experimento: directo-s1-t0.7-20260713


## 2 · Datos

Las entradas (texto libre de los padres) de la semana elegida, con el contexto del paciente (edad calculada a la fecha de la observación, sexo, quién informa).

In [2]:
instrumento = json.load(open(INSTRUMENTO, encoding="utf-8"))
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

con = sqlite3.connect(RUTA_BD)

entradas = pd.read_sql(
    '''
    SELECT e.id_entrada, e.id_paciente, c.rol AS informante, e.fecha,
           p.fecha_nacimiento, p.sexo, e.texto
    FROM entrada e
    JOIN paciente p USING (id_paciente)
    JOIN cuidador c ON c.id_cuidador = e.id_cuidador
    JOIN referencia_sintetica r USING (id_entrada)
    WHERE r.semana = ?
    ORDER BY e.id_paciente
    ''',
    con, params=[SEMANA],
)
if PACIENTES:
    entradas = entradas[entradas["id_paciente"].isin(PACIENTES)]


def calcular_edad(nacimiento, observacion):
    nac = pd.to_datetime(nacimiento).date()
    obs = pd.to_datetime(observacion).date()
    return obs.year - nac.year - ((obs.month, obs.day) < (nac.month, nac.day))


entradas["edad"] = [
    calcular_edad(n, f) for n, f in zip(entradas["fecha_nacimiento"], entradas["fecha"])
]

print(f"Semana {SEMANA}: {len(entradas)} entradas de {entradas['id_paciente'].nunique()} pacientes")
entradas[["id_entrada", "id_paciente", "informante", "edad", "sexo", "texto"]].head()

Instrumento: BRIEF-2 Familia (63 ítems)
Semana 1: 30 entradas de 30 pacientes


,id_entrada,id_paciente,informante,edad,sexo,texto
0,1,PAC001,madre,7,masculino,"Hoy ha sido un día horrible, la verdad. Marco ..."
1,25,PAC002,madre,11,femenino,"Hola, soy la madre de Lucía. Nos dijeron que t..."
2,49,PAC003,padre,15,masculino,Soy el padre de Alejandro. La psiquiatra nos h...
3,73,PAC004,madre,6,femenino,"Somos los padres de Sofía, tiene 6 años. Esta ..."
4,97,PAC005,madre,8,masculino,Soy la madre de Diego. Diego vive conmigo de l...


## 3 · Prompts

El prompt de sistema se construye desde el instrumento (`brief2.json`): catálogo de ítems, escalas y niveles de alerta. El de usuario lleva el contexto del paciente y su texto.

In [3]:
COMILLAS = '"' * 3  # delimitador del texto del padre dentro del prompt


def construir_prompt_sistema(instrumento):
    catalogo = "\n".join(
        f"  {it['id']}: [{it['escala']}] {it['texto']}" for it in instrumento["items"]
    )
    escalas = "\n".join(f"  - {e}: {d}" for e, d in instrumento["escalas"].items())
    n = instrumento["niveles_alerta"]
    return f'''Eres un {instrumento["rol_anotador"]}.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
{instrumento["nombre"]}.

## CATÁLOGO DE ÍTEMS ({len(instrumento["items"])} ítems)
{catalogo}

## ESCALAS
{escalas}

## NIVELES DE ALERTA
{" | ".join(n)}

## INSTRUCCIONES DE SALIDA
Responde ÚNICAMENTE con un objeto JSON válido, sin texto antes ni después, sin markdown.
Estructura requerida:
{{
  "items_detectados": [lista de números de ítem observables en el texto],
  "escalas_afectadas": [lista de escalas correspondientes],
  "nivel_alerta": "{n[0]}|{n[1]}|{n[2]}",
  "nota_clinica": "resumen clínico de 1-3 frases para el médico",
  "justificacion": "explicación del razonamiento (para auditoría)"
}}'''


def construir_prompt_usuario(e):
    return f'''## CONTEXTO DEL PACIENTE
- Edad: {e.edad} años
- Sexo: {e.sexo}
- Informante: {e.informante}

## TEXTO DEL PADRE/MADRE
{COMILLAS}{e.texto}{COMILLAS}

Analiza el texto y genera el JSON de anotación clínica.'''


prompt_sistema = construir_prompt_sistema(instrumento)
print(prompt_sistema[:400] + "\n[...]")

Eres un psicólogo clínico infantil especializado en TDAH y en el instrumento BRIEF-2.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
BRIEF-2 Familia.

## CATÁLOGO DE ÍTEMS (63 ítems)
  1: [inhibicion] Es inquieto o inquieta.
  2: [flexibilidad] Se resiste o le cuesta acept
[...]


## 4 · Backend Directo


In [4]:
import re

import requests


def extraer_json(texto):
    '''Intenta sacar el primer JSON válido del texto que devuelve el modelo.'''
    if not texto:
        return None
    try:
        return json.loads(texto.strip())
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{.*\}", texto, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    limpio = re.sub(r"```(?:json)?", "", texto).strip()
    try:
        return json.loads(limpio)
    except json.JSONDecodeError:
        return None


def diagnosticar_json(cruda):
    '''Muestra dónde falla el parseo de una respuesta cruda del modelo.'''
    if not cruda:
        print("(sin respuesta_cruda — fila anterior a esta versión del cuaderno)")
        return
    print("[FALLO DE FORMATO] El modelo no devolvió un JSON válido:")
    print(f"Longitud: {len(cruda)} caracteres")
    try:
        json.loads(cruda.strip())
    except json.JSONDecodeError as e:
        print(f"Error en posición {e.pos}: {e.msg}")
        print("--- contexto del error ---")
        print(cruda[max(0, e.pos - 120) : e.pos + 120])
    print("--- final ---")
    print(cruda[-300:])

In [5]:
def anotar(prompt_sistema, prompt_usuario):
    '''Llama al modelo y devuelve (anotacion | None, respuesta_cruda).'''
    payload = {
        "model": MODELO,
        "system": prompt_sistema,
        "prompt": prompt_usuario,
        "stream": False,
        "options": {"temperature": TEMPERATURA, "num_predict": 4096},
    }
    r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=180)
    r.raise_for_status()
    cruda = r.json()["response"]
    return extraer_json(cruda), cruda

## 5 · Una anotación de ejemplo

Antes de lanzar el experimento completo, comprobación si funciona bnien.

In [6]:
ejemplo = entradas.iloc[2]
print(f"Paciente {ejemplo.id_paciente} · {ejemplo.informante} · semana {SEMANA}")
print(f"Texto: {ejemplo.texto[:200]}...\n")

t0 = time.time()
anotacion, cruda = anotar(prompt_sistema, construir_prompt_usuario(ejemplo))
print(f"Latencia: {time.time() - t0:.1f}s\n")

if anotacion is None:
    print("[FALLO DE FORMATO] El modelo no devolvió un JSON válido:")
    print(cruda[:500])
else:
    print("ANOTACIÓN FINAL:")
    print(json.dumps(anotacion, indent=2, ensure_ascii=False))

Paciente PAC003 · padre · semana 1
Texto: Soy el padre de Alejandro. La psiquiatra nos ha pedido que hagamos este seguimiento. Alejandro tiene 15 años, acaba de ser diagnosticado con TDAH combinado. El diagnóstico llegó tarde porque siempre f...

Latencia: 17.9s

ANOTACIÓN FINAL:
{
  "items_detectados": [],
  "escalas_afectadas": [
    "flexibilidad",
    "control_emocional",
    "supervision_conducta"
  ],
  "nivel_alerta": "alto",
  "nota_clinica": "Se observa un patrón de rigidez cognitiva y emocional significativo, manifestado en la fuerte resistencia a aceptar el diagnóstico (negación) y la dificultad para manejar los cambios dentro del entorno familiar. El alto nivel de conflicto parental y la pobre adaptación académica sugieren una desregulación emocional y conductual que impacta gravemente su funcionamiento diario.",
  "justificacion": "El texto no describe ítems específicos numerados, por lo cual la lista está vacía. Sin embargo, el contenido narrativo revela tres áreas clave:

Si falla el parseo, script para documentar donde ha fallado:

In [7]:
if anotacion is None:
    diagnosticar_json(cruda)

Este Backend directo algunas veces presenta fallos de formato que corresponden  a errores triviales de sintaxis (comas finales), recuperables con reparación determinista; los backends de salida estructurada los eliminan por construcción

He incluido un nuevo repo con una reparación de este  fallo por coma final, ya que no es un bug: es un hallazgo. El propósito del cuaderno 01 es ser línea base que justifique los backends estructurados. Gemma emite JSON estilo JavaScript (comas finales son válidas en JS/JSON5, y los modelos entrenados con código las arrastran)

## 6. Experimento

### 6.1. REPETIR EXPERIMENTO MISMO 

Reiniciar el código

In [ ]:
# Borra todas las filas de este experimento y empieza de cero
con.execute("DELETE FROM experimento WHERE codigo = ?", [EXPERIMENTO])
con.commit()
print(f"Borradas filas de '{EXPERIMENTO}'. Listo para relanzar.")

### 6.2. Añadir Experimento: 

Mismo día, código distinto (sin borrar el anterior)

In [ ]:
EXPERIMENTO = f"{BACKEND}-s{SEMANA}-t{TEMPERATURA}-{dt.datetime.now():%Y%m%d-%H%M}"

### 6.3. Experimento

Anota cada entrada de la semana `REPETICIONES`, guarda cada resultado en la tabla `experimento` con el código `EXPERIMENTO`. 

Cada fila guarda también `respuesta_cruda` (texto exacto del modelo) para poder diagnosticar fallos de formato después, sin volver a llamar a Ollama.


In [8]:
con.execute('''
CREATE TABLE IF NOT EXISTS experimento (
    id                INTEGER PRIMARY KEY,
    codigo            TEXT NOT NULL,      -- código del experimento (para comparar)
    creada_en         TEXT NOT NULL,
    backend           TEXT NOT NULL,
    modelo            TEXT NOT NULL,
    temperatura       REAL NOT NULL,
    semana            INTEGER,
    id_paciente       TEXT,
    id_entrada        INTEGER,
    repeticion        INTEGER,
    formato_ok        INTEGER,
    items_detectados  TEXT,               -- JSON: [int]
    escalas_afectadas TEXT,               -- JSON: [str]
    nivel_alerta      TEXT,
    nota_clinica      TEXT,
    justificacion     TEXT,
    latencia_s        REAL,
    respuesta_cruda   TEXT                -- texto bruto del modelo (auditoría)
)''')
columnas = {fila[1] for fila in con.execute("PRAGMA table_info(experimento)")}
if "respuesta_cruda" not in columnas:
    con.execute("ALTER TABLE experimento ADD COLUMN respuesta_cruda TEXT")
con.commit()

total = len(entradas) * REPETICIONES
print(f"Experimento '{EXPERIMENTO}': {len(entradas)} entradas × {REPETICIONES} repeticiones "
      f"= {total} llamadas al modelo")

hechas = 0
for _, e in entradas.iterrows():
    prompt_usuario = construir_prompt_usuario(e)
    for rep in range(REPETICIONES):
        t0 = time.time()
        anotacion, cruda = anotar(prompt_sistema, prompt_usuario)
        latencia = time.time() - t0
        ok = anotacion is not None
        a = anotacion or {}
        con.execute(
            "INSERT INTO experimento (codigo, creada_en, backend, modelo, temperatura, "
            "semana, id_paciente, id_entrada, repeticion, formato_ok, items_detectados, "
            "escalas_afectadas, nivel_alerta, nota_clinica, justificacion, latencia_s, "
            "respuesta_cruda) "
            "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (EXPERIMENTO, dt.datetime.now().isoformat(timespec="seconds"), BACKEND,
             MODELO, TEMPERATURA, SEMANA, e.id_paciente, int(e.id_entrada), rep,
             int(ok), json.dumps(a.get("items_detectados", [])),
             json.dumps(a.get("escalas_afectadas", [])), a.get("nivel_alerta"),
             a.get("nota_clinica"), a.get("justificacion"), latencia, cruda),
        )
        con.commit()
        hechas += 1
        estado = "ok" if ok else "FALLO DE FORMATO"
        print(f"  [{hechas:>3}/{total}] {e.id_paciente} rep {rep + 1} → {estado} ({latencia:.1f}s)")

print(f"\nGuardado en la tabla `experimento` con codigo = '{EXPERIMENTO}'")

Experimento 'directo-s1-t0.7-20260713': 30 entradas × 3 repeticiones = 90 llamadas al modelo
  [  1/90] PAC001 rep 1 → ok (10.7s)
  [  2/90] PAC001 rep 2 → ok (12.0s)
  [  3/90] PAC001 rep 3 → FALLO DE FORMATO (11.7s)
  [  4/90] PAC002 rep 1 → ok (10.2s)
  [  5/90] PAC002 rep 2 → ok (9.2s)
  [  6/90] PAC002 rep 3 → ok (10.6s)
  [  7/90] PAC003 rep 1 → ok (10.2s)
  [  8/90] PAC003 rep 2 → ok (8.2s)
  [  9/90] PAC003 rep 3 → ok (10.5s)
  [ 10/90] PAC004 rep 1 → ok (10.1s)
  [ 11/90] PAC004 rep 2 → ok (10.2s)
  [ 12/90] PAC004 rep 3 → ok (10.2s)
  [ 13/90] PAC005 rep 1 → ok (9.6s)
  [ 14/90] PAC005 rep 2 → ok (9.7s)
  [ 15/90] PAC005 rep 3 → ok (9.5s)
  [ 16/90] PAC006 rep 1 → ok (8.4s)
  [ 17/90] PAC006 rep 2 → ok (10.0s)
  [ 18/90] PAC006 rep 3 → ok (8.6s)
  [ 19/90] PAC007 rep 1 → ok (10.0s)
  [ 20/90] PAC007 rep 2 → ok (9.8s)
  [ 21/90] PAC007 rep 3 → ok (8.8s)
  [ 22/90] PAC008 rep 1 → ok (8.2s)
  [ 23/90] PAC008 rep 2 → ok (6.9s)
  [ 24/90] PAC008 rep 3 → ok (6.9s)
  [ 25/90] PAC009

## 7 · Resultados

- `formato_ok`: fracción de salidas que fueron JSON válido.
- `acuerdo_nivel` (0–1): fracción de repeticiones que coincide con el nivel de alerta más frecuente de ese paciente. 1.0 = el modelo dice siempre lo mismo.
- `latencia_media`: segundos por anotación.

In [9]:
df = pd.read_sql(
    "SELECT * FROM experimento WHERE codigo = ?", con, params=[EXPERIMENTO]
)
print(f"{len(df)} anotaciones del experimento '{EXPERIMENTO}'\n")


def acuerdo_modal(niveles):
    '''Fracción de repeticiones que coincide con el nivel más frecuente.'''
    s = niveles.dropna()
    return round(s.value_counts().iloc[0] / len(s), 2) if len(s) else None


resumen = df.groupby("id_paciente").agg(
    repeticiones=("repeticion", "count"),
    formato_ok=("formato_ok", "mean"),
    acuerdo_nivel=("nivel_alerta", acuerdo_modal),
    latencia_media=("latencia_s", "mean"),
).round(2)

print(f"Formato válido: {df['formato_ok'].mean():.0%}")
print(f"Acuerdo medio del nivel de alerta entre repeticiones: {resumen['acuerdo_nivel'].mean():.2f}")
print(f"Latencia media por anotación: {df['latencia_s'].mean():.1f}s\n")
resumen

90 anotaciones del experimento 'directo-s1-t0.7-20260713'

Formato válido: 97%
Acuerdo medio del nivel de alerta entre repeticiones: 0.88
Latencia media por anotación: 8.7s



,repeticiones,formato_ok,acuerdo_nivel,latencia_media
id_paciente,,,,
PAC001,3,0.67,1.00,11.45
PAC002,3,1.00,0.67,10.01
PAC003,3,1.00,0.67,9.60
PAC004,3,1.00,1.00,10.16
PAC005,3,1.00,1.00,9.58
PAC006,3,1.00,1.00,9.00
PAC007,3,1.00,0.67,9.53
PAC008,3,1.00,1.00,7.30
PAC009,3,0.67,1.00,8.89


In [10]:
# Las anotaciones de un paciente concreto, repetición a repetición
UN_PACIENTE = df["id_paciente"].iloc[0]  

detalle = df[df["id_paciente"] == UN_PACIENTE]
print (UN_PACIENTE)
for _, fila in detalle.iterrows():
    print(f"— repetición {fila.repeticion}: nivel={fila.nivel_alerta} "
          f"items={fila.items_detectados}")
    print(f"  nota: {fila.nota_clinica}\n")

PAC001
— repetición 0: nivel=alto items=[1, 3, 4]
  nota: El paciente muestra un patrón de conducta que sugiere dificultades significativas en la autorregulación, manifestadas por hiperactividad motora (saltar durante el desayuno) e impulsividad física notable. Se observa además una dificultad clara en la memoria de trabajo para seguir instrucciones multi-paso y déficits en la conciencia del impacto de sus acciones sobre su entorno social.

— repetición 1: nivel=alto items=[1, 3, 9, 10, 25, 26, 30]
  nota: El niño presenta un cuadro clínico con alta sintomatología de hiperactividad, manifestada por inquietud constante y dificultad para permanecer sentado. Se observa una marcada desregulación conductual e impulsiva (empujones al correr) junto a déficits significativos en la memoria de trabajo y en el seguimiento secuencial de instrucciones. Su capacidad para mantener la atención en tareas complejas o terminar recados es limitada, lo que requiere intervención multidisciplinar urgente.

—

In [11]:
# Diagnosticar fallos guardados (lee respuesta_cruda de la BD, sin volver a llamar al modelo)
fallos = pd.read_sql(
    """
    SELECT id, id_paciente, id_entrada, repeticion, latencia_s, creada_en, respuesta_cruda
    FROM experimento
    WHERE codigo = ? AND formato_ok = 0
    ORDER BY id
    """,
    con,
    params=[EXPERIMENTO],
)
print(f"{len(fallos)} fallos registrados en '{EXPERIMENTO}'\n")

for _, f in fallos.iterrows():
    print("=" * 70)
    print(f"BD id={f.id} · {f.id_paciente} · entrada {f.id_entrada} · rep {f.repeticion}")
    print(f"Guardado: {f.creada_en} · latencia: {f.latencia_s:.1f}s")
    diagnosticar_json(f.respuesta_cruda)
    print()

3 fallos registrados en 'directo-s1-t0.7-20260713'

BD id=183 · PAC001 · entrada 1 · rep 2
Guardado: 2026-07-13T08:54:56 · latencia: 11.7s
[FALLO DE FORMATO] El modelo no devolvió un JSON válido:
Longitud: 1209 caracteres
Error en posición 705: Expecting ',' delimiter
--- contexto del error ---
 trabajo.",
  "justificacion": "Se detectaron cuatro ítems clave basados en el relato materno: (1) Inquietud excesiva ("saltando de la silla literalmente cada dos por tres"), afectando la escala Inhibición. (3) Dificultad para recordar secu
--- final ---
. (20) Problemas en la conciencia social, ya que el paciente empujó sin darse cuenta del efecto ("no se entera de nada de lo que hace"), afectando Supervisión Conducta. (39) Actúa fuera de control o alocado, reflejado en correr por los pasillos y empujar, lo cual refuerza la esfera de Inhibición."
}

BD id=206 · PAC009 · entrada 193 · rep 1
Guardado: 2026-07-13T08:58:29 · latencia: 8.3s
[FALLO DE FORMATO] El modelo no devolvió un JSON válido:
Lo

## 8 · Siguiente paso

Comparar este experimento con los de los otros backends (u otros parámetros) en `05_comparacion_experimentos.ipynb`, usando los códigos de experimento.